In [ ]:
import boto3
import botocore
from IPython.core.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from teeplot import teeplot as tp


In [ ]:
from dishpylib.pyhelpers import print_runtime


In [ ]:
print_runtime()


In [ ]:
teeplot_subdir = "2025-11-25-external-timeshift"


In [ ]:
df_all = []
for what, prefix in [
    ("nobb", "endeavor=16/external-competitions/stage=2+what=collated/"),
    ("selfbb", "endeavor=16/external-competitions-focalbb/stage=2+what=collated/"),
    ("selfbb33", "endeavor=16/external-competitions-bg33-focalbb/stage=2+what=collated/"),
    ("selfbb40", "endeavor=16/external-competitions-bg40-focalbb/stage=2+what=collated/"),
    ("selfbb50", "endeavor=16/external-competitions-bg50-focalbb/stage=2+what=collated/"),
    ("selfbb66", "endeavor=16/external-competitions-bg66-focalbb/stage=2+what=collated/"),
    ("selfbb75", "endeavor=16/external-competitions-bg75-focalbb/stage=2+what=collated/"),
    ("truebb", "endeavor=16/external-competitions-focalbb-true/stage=2+what=collated/"),
    ("selfbb90", "endeavor=16/external-competitions-bg90-focalbb/stage=2+what=collated/"),
    ("selfbb99", "endeavor=16/external-competitions-bg99-focalbb/stage=2+what=collated/"),
]:
    s3_handle = boto3.resource(
        's3',
        region_name="us-east-2",
        config=botocore.config.Config(
            signature_version=botocore.UNSIGNED,
        ),
    )
    bucket_handle = s3_handle.Bucket("prq49")


    competitions = bucket_handle.objects.filter(Prefix=prefix)
    dfs = [
        pd.read_csv(
          f's3://prq49/{competitions.key}',
        )
        for competitions in competitions
    ]
    print(len(dfs))
    df = pd.concat(dfs, ignore_index=True)
    df["kind"] = what

    df_all.append(df)

df = pd.concat(df_all, ignore_index=True)


In [ ]:
dfx = df.loc[
    (df["Root ID"] == 0),
    ["Competition Stint", "Focal Prevalence", "kind"]
]
display(dfx["Focal Prevalence"])

dfx["Focal Prevalence Rounded"] = dfx["Focal Prevalence"].round()
display(dfx["Focal Prevalence Rounded"])


with tp.teed(
    sns.lineplot,
    data=dfx[
        dfx["kind"] != "truebb"
    ],
    x="Competition Stint",
    y="Focal Prevalence",
    hue="kind",
    teeplot_subdir=teeplot_subdir,
) as ax:
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

    plt.axhline(y=0.5, color='r', linestyle='--')
    plt.ylim(0, 1)


In [ ]:
dfx = df.loc[
    (df["Root ID"] == 0),
    ["Competition Stint", "Focal Prevalence", "kind"]
]
display(dfx["Focal Prevalence"])

for kind in [
    "selfbb33",
    "selfbb40",
    "selfbb50",
    "selfbb75",
    "selfbb66",
    "selfbb99",
    "selfbb90",
]:
    with tp.teed(
        sns.lineplot,
        data=dfx[
            dfx["kind"].isin(["nobb", kind, "selfbb"])
        ],
        x="Competition Stint",
        y="Focal Prevalence",
        hue="kind",
        teeplot_subdir=teeplot_subdir,
    ) as ax:
        sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

        plt.axhline(y=0.5, color='r', linestyle='--')
        plt.ylim(0, 1)
